<a href="https://colab.research.google.com/github/robotics-hana/COMP0173_T1_25/blob/main/Preprocessing%20Iraq%20Marshes%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
!mkdir -p "/content/drive/MyDrive/Iraq_Marshes/images"
!mkdir -p "/content/drive/MyDrive/Iraq_Marshes/masks"
!mv "/content/drive/MyDrive/Iraq_Marshes/"*"_s2_4band"*.tif* "/content/drive/MyDrive/Iraq_Marshes/images/" 2>/dev/null || true
!mv "/content/drive/MyDrive/Iraq_Marshes/"*"_gsw_water"*.tif* "/content/drive/MyDrive/Iraq_Marshes/masks/" 2>/dev/null || true

!echo "IMAGES:"; ls "/content/drive/MyDrive/Iraq_Marshes/images" | head -n 20
!echo "MASKS:";  ls "/content/drive/MyDrive/Iraq_Marshes/masks"  | head -n 20

image_dir = "/content/drive/MyDrive/Iraq_Marshes/images"
mask_dir  = "/content/drive/MyDrive/Iraq_Marshes/masks"
!mkdir -p "/content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg"


IMAGES:
central_marshes_2021_s2_4band.tif
hammar_marshes_2021_s2_4band.tif
hawizeh_marshes_2021_s2_4band.tif
MASKS:
central_marshes_2021_gsw_water.tif
hammar_marshes_2021_gsw_water.tif
hawizeh_marshes_2021_gsw_water.tif


In [8]:

# ---------------------------
# 0) PATHS + SETTINGS
# ---------------------------
GEE_IMG_DIR = "/content/drive/MyDrive/Iraq_Marshes/images"
GEE_MSK_DIR = "/content/drive/MyDrive/Iraq_Marshes/masks"
OUT_ROOT    = "/content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg"

TILE = 512
SEED = 42

ONLY_YEAR = 2021  # or None

TARGET_TRAIN = 250
TARGET_VAL   = 100
TARGET_TEST  = 20

WATER_HEAVY_FRAC = 0.15
P_EMPTY = 0.33
P_MIXED = 0.34
P_HEAVY = 0.33

# ✅ Marsh-wise splits (NO LEAKAGE)
TRAIN_MARSHES = {"central_marshes"}
VAL_MARSHES   = {"hammar_marshes"}
TEST_MARSHES  = {"hawizeh_marshes"}

random.seed(SEED)
np.random.seed(SEED)

# ---------------------------
# 1) HELPERS
# ---------------------------
def make_dirs(root: str) -> None:
    for split in ["training", "validation", "test"]:
        os.makedirs(os.path.join(root, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(root, split, "masks"), exist_ok=True)
        os.makedirs(os.path.join(root, split, "rgb_png"), exist_ok=True)

def align_mask_to_image(img_path: str, mask_path: str):
    with rasterio.open(img_path) as src_img:
        img = src_img.read().astype(np.float32)
        img_transform = src_img.transform
        img_crs = src_img.crs
        img_h, img_w = src_img.height, src_img.width

    img = np.nan_to_num(img)

    with rasterio.open(mask_path) as src_msk:
        msk = src_msk.read(1).astype(np.float32)
        msk = np.nan_to_num(msk)

        same_grid = (
            src_msk.crs == img_crs and
            src_msk.transform == img_transform and
            src_msk.width == img_w and
            src_msk.height == img_h
        )

        if same_grid:
            mask_resampled = msk
        else:
            mask_resampled = np.zeros((img_h, img_w), dtype=np.float32)
            reproject(
                source=msk,
                destination=mask_resampled,
                src_transform=src_msk.transform,
                src_crs=src_msk.crs,
                dst_transform=img_transform,
                dst_crs=img_crs,
                resampling=Resampling.nearest
            )

    img_arr  = np.transpose(img, (1, 2, 0))
    mask_arr = (mask_resampled > 0.5).astype(np.uint8)
    return img_arr, mask_arr

def per_tile_minmax(x):
    mn, mx = float(x.min()), float(x.max())
    if mx <= mn:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn)

def water_fraction(msk):
    return float(msk.mean())

def parse_marsh_year_and_base(s2_path):
    fname = os.path.basename(s2_path)
    stem = os.path.splitext(fname)[0]
    base = stem.replace("_s2_4band", "")
    parts = base.split("_")
    year = int(parts[-1])
    marsh = "_".join(parts[:-1])
    return marsh, year, base

def bucket_tiles(tiles):
    empty = [t for t in tiles if t["wf"] == 0.0]
    heavy = [t for t in tiles if t["wf"] >= WATER_HEAVY_FRAC]
    mixed = [t for t in tiles if 0.0 < t["wf"] < WATER_HEAVY_FRAC]
    return empty, mixed, heavy

def sample_k(lst, k):
    random.shuffle(lst)
    return lst[:min(k, len(lst))]

def build_split_balanced(pool, n):
    empty, mixed, heavy = bucket_tiles(pool)

    n_e = int(P_EMPTY * n)
    n_m = int(P_MIXED * n)
    n_h = n - n_e - n_m

    chosen = (
        sample_k(empty, n_e) +
        sample_k(mixed, n_m) +
        sample_k(heavy, n_h)
    )

    if len(chosen) < n:
        remaining = [t for t in pool if t not in chosen]
        chosen += sample_k(remaining, n - len(chosen))

    random.shuffle(chosen)
    return chosen

def summarise_split(name, tiles):
    wfs = np.array([t["wf"] for t in tiles])
    marshes = sorted(set(t["marsh"] for t in tiles))
    years   = sorted(set(t["year"] for t in tiles))
    print(
        f"{name}: n={len(tiles)} | marshes={marshes} | years={years} | "
        f"wf_mean={wfs.mean():.3f}"
    )

# ---------------------------
# 2) RGB PREVIEW
# ---------------------------
def rgb_preview_uint8(img4):
    rgb = img4[..., [2,1,0]].astype(np.float32)
    out = np.zeros_like(rgb)
    for c in range(3):
        p2, p98 = np.percentile(rgb[..., c], (2, 98))
        out[..., c] = np.clip((rgb[..., c] - p2) / (p98 - p2 + 1e-6), 0, 1)
    return (out * 255).astype(np.uint8)

# ---------------------------
# 3) SAVE
# ---------------------------
def save_tiles(tiles, split):
    for t in tiles:
        np.save(f"{OUT_ROOT}/{split}/images/{t['id']}.npy",
                per_tile_minmax(t["img"]).astype(np.float32))
        np.save(f"{OUT_ROOT}/{split}/masks/{t['id']}.npy",
                t["msk"][..., None].astype(np.uint8))
        imageio.imwrite(f"{OUT_ROOT}/{split}/rgb_png/{t['id']}.png",
                         rgb_preview_uint8(t["img"]))

# ---------------------------
# 4) COLLECT TILES
# ---------------------------
def collect_tiles_grouped_by_marsh():
    tiles_by_marsh = defaultdict(list)
    for s2_path in glob.glob(os.path.join(GEE_IMG_DIR, "*_s2_4band.tif*")):
        marsh, year, base = parse_marsh_year_and_base(s2_path)
        if ONLY_YEAR and year != ONLY_YEAR:
            continue

        mask_path = os.path.join(GEE_MSK_DIR, base + "_gsw_water.tif")
        if not os.path.exists(mask_path):
            continue

        print("Reading + aligning:", base)
        img, msk = align_mask_to_image(s2_path, mask_path)

        H, W, _ = img.shape
        for r in range(0, H - TILE + 1, TILE):
            for c in range(0, W - TILE + 1, TILE):
                tile_id = f"{base}_r{r//TILE:03d}_c{c//TILE:03d}"
                img_t = img[r:r+TILE, c:c+TILE]
                msk_t = msk[r:r+TILE, c:c+TILE]
                tiles_by_marsh[marsh].append({
                    "id": tile_id,
                    "img": img_t,
                    "msk": msk_t,
                    "marsh": marsh,
                    "year": year,
                    "wf": water_fraction(msk_t)
                })
    return tiles_by_marsh

# ---------------------------
# 5) BUILD SPLITS
# ---------------------------
def build_splits_from_marshes(tiles_by_marsh):
    train = [t for m in TRAIN_MARSHES for t in tiles_by_marsh[m]]
    val   = [t for m in VAL_MARSHES   for t in tiles_by_marsh[m]]
    test  = [t for m in TEST_MARSHES  for t in tiles_by_marsh[m]]

    return (
        build_split_balanced(train, TARGET_TRAIN),
        build_split_balanced(val,   TARGET_VAL),
        build_split_balanced(test,  TARGET_TEST),
    )

# ---------------------------
# 6) RUN
# ---------------------------
make_dirs(OUT_ROOT)

tiles_by_marsh = collect_tiles_grouped_by_marsh()
print("Marshes found:", list(tiles_by_marsh.keys()))

train, val, test = build_splits_from_marshes(tiles_by_marsh)

summarise_split("TRAIN", train)
summarise_split("VAL",   val)
summarise_split("TEST",  test)

save_tiles(train, "training")
save_tiles(val,   "validation")
save_tiles(test,  "test")

print("✅ Done. Saved to:", OUT_ROOT)

Reading + aligning: central_marshes_2021
Reading + aligning: hammar_marshes_2021
Reading + aligning: hawizeh_marshes_2021
Marshes found: ['central_marshes', 'hammar_marshes', 'hawizeh_marshes']
TRAIN: n=143 | marshes=['central_marshes'] | years=[2021] | wf_mean=0.112
VAL: n=99 | marshes=['hammar_marshes'] | years=[2021] | wf_mean=0.059
TEST: n=20 | marshes=['hawizeh_marshes'] | years=[2021] | wf_mean=0.212
✅ Done. Saved to: /content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg


In [ ]:
!ls -lah "/content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg"
!ls -lah "/content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg/validation/images" | head
!ls -lah "/content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg/test/images" | head


total 12K
drwx------ 4 root root 4.0K Jan 17 16:27 test
drwx------ 4 root root 4.0K Jan 17 16:27 training
drwx------ 4 root root 4.0K Jan 17 16:27 validation
total 401M
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r000_c002.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r001_c000.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r001_c008.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r001_c011.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r001_c021.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r001_c024.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r002_c032.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r002_c034.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r002_c035.npy
total 81M
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r000_c018.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021